# Part 6: Plotly — Interactive Data Visualization
**Course Notes Reference:** Part 13

Topics covered:
1. `plotly.express` (`px`): Line, Bar, Scatter (with `hover_name`, `size`, `color`), Histogram, Box/Violin, Donut, Heatmap (`px.imshow`), Treemap, Sunburst, Scatter Matrix
2. Animated scatter plot (`animation_frame='Year'`)
3. `plotly.graph_objects` (`go`) & `make_subplots` for full control
4. Exporting standalone interactive HTML (`fig.write_html(...)`)


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("vgsales.csv").dropna(subset=["Year", "Publisher"]).copy()
df["Year"] = df["Year"].astype(int)


## 1. Interactive Scatter, Bar, Line & Donut Charts


In [2]:
# Interactive Scatter of top 500 games
fig = px.scatter(
    df.head(500), x="NA_Sales", y="EU_Sales",
    color="Genre", size="Global_Sales",
    hover_name="Name", hover_data=["Platform", "Year", "Publisher"],
    title="Top 500 Games: NA vs EU Sales (Interactive)"
)
fig.update_layout(template="plotly_white", width=900, height=500,
                  xaxis_title="North America Sales (M $)", yaxis_title="Europe Sales (M $)")
fig.show()


<Plotly Figure: Top 500 Games: NA vs EU Sales (Interactive)>

In [3]:
# Grouped Bar: Top 5 Genres Total Sales by Top 3 Platforms
top5_g = df["Genre"].value_counts().head(5).index
top3_p = df["Platform"].value_counts().head(3).index
sub = df[df["Genre"].isin(top5_g) & df["Platform"].isin(top3_p)]
agg = sub.groupby(["Genre", "Platform"], as_index=False)["Global_Sales"].sum()

fig = px.bar(agg, x="Genre", y="Global_Sales", color="Platform", barmode="group",
             title="Total Global Sales by Genre & Platform", template="plotly_white")
fig.show()


<Plotly Figure: Total Global Sales by Genre & Platform>

In [4]:
# Interactive Donut Chart of Regional Sales
region = df[["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]].sum().reset_index()
region.columns = ["Region", "Sales"]

fig = px.pie(region, names="Region", values="Sales", hole=0.4, title="Regional Sales Share (Interactive Donut)")
fig.show()


<Plotly Figure: Regional Sales Share (Interactive Donut)>

## 2. Hierarchy (Treemap & Sunburst), Heatmap & Scatter Matrix


In [5]:
# Treemap: Genre -> Top 5 Platforms
top5_plat = df["Platform"].value_counts().head(5).index
sub_tree = df[df["Platform"].isin(top5_plat)]

fig = px.treemap(
    sub_tree, path=["Genre", "Platform"], values="Global_Sales",
    title="Global Sales Hierarchy: Genre -> Platform (Treemap)"
)
fig.show()


<Plotly Figure: Global Sales Hierarchy: Genre -> Platform (Treemap)>

In [6]:
# Sunburst
fig = px.sunburst(
    sub_tree, path=["Genre", "Platform"], values="Global_Sales",
    title="Global Sales Hierarchy: Genre -> Platform (Sunburst)"
)
fig.show()


<Plotly Figure: Global Sales Hierarchy: Genre -> Platform (Sunburst)>

In [7]:
# Correlation Heatmap with px.imshow
corr = df[["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Interactive Sales Correlation Heatmap")
fig.show()


<Plotly Figure: Interactive Sales Correlation Heatmap>

## 3. Animated Scatter Plot Over Years (`animation_frame`)


In [8]:
anim_df = df[(df["Year"] >= 2000) & (df["Year"] <= 2015)].sort_values("Year")
fig = px.scatter(
    anim_df, x="NA_Sales", y="EU_Sales",
    animation_frame="Year", color="Genre", size="Global_Sales",
    hover_name="Name", range_x=[0, 20], range_y=[0, 15],
    title="NA vs EU Sales Evolution (2000–2015)"
)
fig.update_layout(template="plotly_white")
fig.show()


<Plotly Figure: NA vs EU Sales Evolution (2000–2015)>

## 4. `plotly.graph_objects` Subplots & HTML Export


In [9]:
genre_counts = df["Genre"].value_counts().head(6)
yearly_sales = df.groupby("Year")["Global_Sales"].sum().loc[1990:2015]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Top 6 Genres (Count)", "Annual Global Sales (M $)"))
fig.add_trace(go.Bar(x=genre_counts.index, y=genre_counts.values, name="Games"), row=1, col=1)
fig.add_trace(go.Scatter(x=yearly_sales.index, y=yearly_sales.values, mode="lines+markers", name="Sales"), row=1, col=2)
fig.update_layout(height=420, width=950, title_text="Plotly Graph Objects Subplots", template="plotly_white")
fig.show()

# Save standalone interactive HTML:
# fig.write_html("plotly_dashboard.html")


<Plotly Figure: Plotly Graph Objects Subplots>